# Euler Characteristic and Combinatorics of Parabolic Arrangements

A quick notebook demonstrating that the **orbifold Euler characteristic** (weighted by parabolic stabilizer orders) coincides with the **alternating sum of Betti numbers**.

> J. Cantarero and J. L. León-Medina, *The Topology of Real Parabolic Arrangements*.

---

## The Formula

For a $W$-invariant parabolic arrangement, the Euler characteristic satisfies:

$$\chi(K_{\mathscr{A}}) = \sum_{k \ge 0} (-1)^k \, |C^k| = \sum_{k \ge 0} (-1)^k \, \beta_k$$

where $C^k$ are the $k$-cells in the complement and $\beta_k = \dim H^k$.

The $W$-invariant formula computes $\chi$ as a sum over orbit types $[J]$:

$$\chi(K_{\mathscr{A}}) = |W| \cdot \sum_{[J] \text{ surviving}} \frac{(-1)^{|J|}}{|W_J|}$$

In [1]:
load('../parabolic_arrangements.sage')

## Test 1: $A_2$, Full Permutahedron (No Removal)

With no cells removed, the permutahedron is contractible and $\chi = 1$.

In [2]:
W, Plist, _ = build_W_P('A', 2)
arr = ParabolicArrangement(W, Plist, set())
chi_gen = arr.euler_characteristic()
chi_inv = arr.euler_characteristic_W_invariant(verbose=True)
print(f'A2 full: chi_general={chi_gen}, chi_W_inv={chi_inv}, match={chi_gen == chi_inv}')

  |W| = 6
  Retained J-types: 4
    J=():  |W_J|=1,  N_J=[W:W_J]=6,  (-1)^0/|W_J| = 1
    J=(1,):  |W_J|=2,  N_J=[W:W_J]=3,  (-1)^1/|W_J| = -1/2
    J=(2,):  |W_J|=2,  N_J=[W:W_J]=3,  (-1)^1/|W_J| = -1/2
    J=(1, 2):  |W_J|=6,  N_J=[W:W_J]=1,  (-1)^2/|W_J| = 1/6
  Weighted sum = 1/6
  chi = |W| * sum = 6 * 1/6 = 1
A2 full: chi_general=1, chi_W_inv=1, match=True


## Test 2: $A_3$, 3-Parabolic Arrangement

The $k=3$ arrangement removes hexagonal faces. The expected Betti numbers are $(1, 7, 0)$, giving $\chi = 1 - 7 + 0 = -6$.

In [3]:
W, Plist, _ = build_W_P('A', 3)
Delta = ideal_k_parabolic(W, Plist, k=3)
arr = ParabolicArrangement(W, Plist, Delta)
chi_gen = arr.euler_characteristic()
chi_inv = arr.euler_characteristic_W_invariant(verbose=True)
print(f'A3 k=3: chi_general={chi_gen}, chi_W_inv={chi_inv}, match={chi_gen == chi_inv}')

  |W| = 24
  Retained J-types: 5
    J=():  |W_J|=1,  N_J=[W:W_J]=24,  (-1)^0/|W_J| = 1
    J=(1,):  |W_J|=2,  N_J=[W:W_J]=12,  (-1)^1/|W_J| = -1/2
    J=(2,):  |W_J|=2,  N_J=[W:W_J]=12,  (-1)^1/|W_J| = -1/2
    J=(3,):  |W_J|=2,  N_J=[W:W_J]=12,  (-1)^1/|W_J| = -1/2
    J=(1, 3):  |W_J|=4,  N_J=[W:W_J]=6,  (-1)^2/|W_J| = 1/4
  Weighted sum = -1/4
  chi = |W| * sum = 24 * -1/4 = -6
A3 k=3: chi_general=-6, chi_W_inv=-6, match=True


## Test 3: $B_3$, 3-Parabolic Arrangement

The hyperoctahedral group $B_3$ has $|W| = 48$. The arrangement again removes strata of rank $\ge 3$.

In [4]:
W, Plist, _ = build_W_P('B', 3)
Delta = ideal_k_parabolic(W, Plist, k=3)
arr = ParabolicArrangement(W, Plist, Delta)
chi_gen = arr.euler_characteristic()
chi_inv = arr.euler_characteristic_W_invariant(verbose=True)
print(f'B3 k=3: chi_general={chi_gen}, chi_W_inv={chi_inv}, match={chi_gen == chi_inv}')

  |W| = 48
  Retained J-types: 5
    J=():  |W_J|=1,  N_J=[W:W_J]=48,  (-1)^0/|W_J| = 1
    J=(1,):  |W_J|=2,  N_J=[W:W_J]=24,  (-1)^1/|W_J| = -1/2
    J=(2,):  |W_J|=2,  N_J=[W:W_J]=24,  (-1)^1/|W_J| = -1/2
    J=(3,):  |W_J|=2,  N_J=[W:W_J]=24,  (-1)^1/|W_J| = -1/2
    J=(1, 3):  |W_J|=4,  N_J=[W:W_J]=12,  (-1)^2/|W_J| = 1/4
  Weighted sum = -1/4
  chi = |W| * sum = 48 * -1/4 = -12
B3 k=3: chi_general=-12, chi_W_inv=-12, match=True


## Cross-Check: Euler via Betti Numbers

The most important consistency check: the alternating sum of Betti numbers $\sum (-1)^k \beta_k$ must equal the cell-count Euler characteristic $\sum (-1)^k |C^k|$.

In [5]:
for label, tp, rk, kk in [('A3', 'A', 3, 3), ('B3', 'B', 3, 3)]:
    W, Plist, _ = build_W_P(tp, rk)
    Delta = ideal_k_parabolic(W, Plist, k=kk)
    arr = ParabolicArrangement(W, Plist, Delta)
    betti = [len(arr.cohomology_basis(i, GF(32003))) for i in range(arr.max_grade + 1)]
    chi_betti = sum((-1)**i * b for i, b in enumerate(betti))
    chi_cells = arr.euler_characteristic()
    print(f'{label}: Betti={betti}, chi(Betti)={chi_betti}, chi(cells)={chi_cells}, match={chi_betti == chi_cells}')

A3: Betti=[1, 7, 0], chi(Betti)=-6, chi(cells)=-6, match=True
B3: Betti=[1, 13, 0], chi(Betti)=-12, chi(cells)=-12, match=True


## Summary

| Type | \|W\| | Betti | chi (cells) | chi (W-inv) | chi (Betti) | Match |
|------|-------|-------|-------------|-------------|-------------|-------|
| A2 full | 6 | (1) | 1 | 1 | 1 | ✓ |
| A3 k=3 | 24 | (1, 7, 0) | -6 | -6 | -6 | ✓ |
| B3 k=3 | 48 | computed | computed | computed | computed | ✓ |

The three formulas for chi agree in every case, validating both the cell structure and the orbifold Euler characteristic formula.

---

**Version**: 3.0 | **Updated**: March 2026